#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the ephemeral storage, installs ComfyUI, and configures the SeedVR2 Upscaling node.

import os
import subprocess
from IPython.display import clear_output

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 [@CoinNoin] Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 [@CoinNoin] Installing Dependencies (This takes a moment)...")
!cd {LOCAL_WORKSPACE} && pip install -q -r requirements.txt &> /dev/null

SEEDVR_NODE_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/ComfyUI-SeedVR2_VideoUpscaler")
if not os.path.exists(SEEDVR_NODE_DIR):
    print("🧩 [@CoinNoin] Installing SeedVR2 Processing Nodes...")
    !git clone https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler {SEEDVR_NODE_DIR} &> /dev/null
    !pip install -q -r {SEEDVR_NODE_DIR}/requirements.txt &> /dev/null
else:
    !cd {SEEDVR_NODE_DIR} && git pull &> /dev/null

clear_output()
print("✅ [@CoinNoin] Environment Ready!")

In [ ]:
#@title 2. High-Speed Asset Downloader
#@markdown The **SeedVR2 DiT (3B)** and **VAE** models are fetched automatically using high-speed multi-threading.

import os
import subprocess

WORKSPACE = "/content/ComfyUI"
MODEL_DIR = os.path.join(WORKSPACE, "models", "seedvr2")
os.makedirs(MODEL_DIR, exist_ok=True)

DIT_URL = "https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/seedvr2_ema_3b_fp16.safetensors"
VAE_URL = "https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/ema_vae_fp16.safetensors"

print("⚡ [@CoinNoin] Configuring Aria2c Accelerator...")
subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def download(url, dest_dir, filename, desc):
    dest = os.path.join(dest_dir, filename)
    if os.path.exists(dest) and os.path.getsize(dest) > 10_000_000:
        print(f"   ✅ Already Cached: {desc}")
        return

    print(f"   📥 Fetching: {desc}...")
    cmd = [
        "aria2c", "--console-log-level=error", "--summary-interval=10",
        "-c", "-x", "16", "-s", "16", "-k", "1M",
        "-o", filename, url, "-d", dest_dir
    ]
    subprocess.run(cmd, check=True)
    print(f"   ✅ Downloaded: {desc}")

print("\n📥 [@CoinNoin] Commencing Model Downloads...")
download(DIT_URL, MODEL_DIR, "seedvr2_ema_3b_fp16.safetensors", "SeedVR2 DiT (3B)")
download(VAE_URL, MODEL_DIR, "ema_vae_fp16.safetensors", "SeedVR2 VAE")

print("\n✅ [@CoinNoin] All core assets secured!")

In [ ]:
#@title 3. SeedVR2 Image Upscaler
#@markdown Upload an image and configure the upscaling parameters. Pure upscaling – no prompt generation required.

# --- Upload Controls ---
UPLOAD_INPUT_IMAGE = True #@param {type:"boolean"}
#@markdown *If checked, Colab will prompt you to upload an image. Uncheck to re-run the previous image with new settings.*

# --- Upscale Settings ---
UPSCALE_FACTOR = 2.0 #@param {type:"slider", min:1.0, max:4.0, step:0.5}
#@markdown *Multiplies the shortest edge of your image by this factor.*

COLOR_CORRECTION = "lab" #@param ["lab", "none"]
UPSCALE_ALPHA = False #@param {type:"boolean"}
#@markdown *Check this to upscale images with transparency (RGBA).*

INPUT_NOISE_SCALE = 0.0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
LATENT_NOISE_SCALE = 0.0 #@param {type:"slider", min:0.0, max:0.1, step:0.01}
SEED = 0 #@param {type:"integer"}

# --- Direct Download ---
AUTO_DOWNLOAD = True #@param {type:"boolean"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from PIL import Image
from google.colab import files
from IPython.display import display, Image as IPImage, clear_output, HTML

WORKSPACE = "/content/ComfyUI"
INPUT_DIR = os.path.join(WORKSPACE, "input")
os.makedirs(INPUT_DIR, exist_ok=True)
os.chdir(WORKSPACE)

if SEED == 0:
    SEED = random.randint(1, 2**31)

print("🔌 [@CoinNoin] Checking ComfyUI Server Status...")
def start_server():
    req = urllib.request.Request("http://127.0.0.1:8188")
    try:
        urllib.request.urlopen(req)
        print("   🟢 Server is already running.")
    except:
        print("   🚀 Starting ComfyUI Server in background...")
        subprocess.Popen([sys.executable, "main.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(req)
                print("   🟢 Server is now up and ready!")
                break
            except:
                time.sleep(2)

start_server()
clear_output()

# Handle Uploads
input_image_filename = "seedvr2_input.png"
if UPLOAD_INPUT_IMAGE:
    print("📤 [@CoinNoin] Please upload your SOURCE image to upscale:")
    uploaded = files.upload()
    if uploaded:
        source_name = list(uploaded.keys())[0]
        os.replace(source_name, os.path.join(INPUT_DIR, input_image_filename))
        print("   ✅ Source image saved!")
    else:
        print("   ❌ No image uploaded. Will attempt to use cached image.")
else:
    print("   ℹ️ Using previously uploaded source image.")

input_path = os.path.join(INPUT_DIR, input_image_filename)
if not os.path.exists(input_path):
    raise RuntimeError("❌ No input image found in storage! Please check 'UPLOAD_INPUT_IMAGE' and run again.")

# --- Display Input Thumbnail ---
print("\n🖼️ [@CoinNoin] Input Image Preview:")
display(IPImage(filename=input_path, width=256))

# Calculate Resolution Target based on image size and upscale factor
try:
    with Image.open(input_path) as img:
        w, h = img.size
        shortest = min(w, h)
        target_res = int(shortest * UPSCALE_FACTOR)
        # Ensure resolution is an even number
        target_res = target_res // 2 * 2
        print(f"\n📊 [@CoinNoin] Image Stats:")
        print(f"   Original Size: {w}x{h}")
        print(f"   Target Shortest Edge: {target_res}px")
except Exception as e:
    raise RuntimeError(f"❌ Could not read the uploaded image: {e}")

# Brand visibility prefix in filename
BRAND_PREFIX = "@CoinNoin_SeedVR2"

print(f"\n\033[94m➜ [@CoinNoin] Upscale Details | Factor: {UPSCALE_FACTOR}x | Color: {COLOR_CORRECTION.upper()} | Seed: {SEED}\033[0m")

# Build SeedVR2 Architecture
workflow = {
    "load_image": {
        "class_type": "LoadImage",
        "inputs": {"image": input_image_filename}
    },
    "load_dit": {
        "class_type": "SeedVR2LoadDiTModel",
        "inputs": {
            "model": "seedvr2_ema_3b_fp16.safetensors",
            "device": "cuda:0",
            "blocks_to_swap": 0,
            "swap_io_components": False,
            "offload_device": "cpu",
            "cache_model": False,
            "attention_mode": "sdpa"
        }
    },
    "load_vae": {
        "class_type": "SeedVR2LoadVAEModel",
        "inputs": {
            "model": "ema_vae_fp16.safetensors",
            "device": "cuda:0",
            "encode_tiled": True,
            "encode_tile_size": 1024,
            "encode_tile_overlap": 128,
            "decode_tiled": True,
            "decode_tile_size": 1024,
            "decode_tile_overlap": 128,
            "tile_debug": "false",
            "offload_device": "cpu",
            "cache_model": False
        }
    },
    "upscaler": {
        "class_type": "SeedVR2VideoUpscaler",
        "inputs": {
            "image": ["load_image", 0],
            "dit": ["load_dit", 0],
            "vae": ["load_vae", 0],
            "seed": SEED,
            "resolution": target_res,
            "max_resolution": 0,
            "batch_size": 1,
            "uniform_batch_size": False,
            "color_correction": COLOR_CORRECTION,
            "temporal_overlap": 0,
            "prepend_frames": 0,
            "input_noise_scale": INPUT_NOISE_SCALE,
            "latent_noise_scale": LATENT_NOISE_SCALE,
            "offload_device": "cpu",
            "enable_debug": False
        }
    },
    "save": {
        "class_type": "SaveImage",
        "inputs": {
            "filename_prefix": BRAND_PREFIX,
            "images": ["upscaler", 0]
        }
    }
}

if UPSCALE_ALPHA:
    workflow["join_alpha"] = {
        "class_type": "JoinImageWithAlpha",
        "inputs": {
            "image": ["load_image", 0],
            "alpha": ["load_image", 1]
        }
    }
    workflow["upscaler"]["inputs"]["image"] = ["join_alpha", 0]
    print("✨ [@CoinNoin] Alpha channel upscaling enabled.")

p = {"prompt": workflow}
data = json.dumps(p).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)

print("📥 Submitting workflow to API...")
try:
    response = urllib.request.urlopen(req)
    prompt_id = json.loads(response.read())['prompt_id']
except Exception as e:
    print(f"❌ API Error: {e}")
    raise

print("✨ Processing and Upscaling (Check ComfyUI server logs if stuck)...")
while True:
    try:
        history_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        history_res = urllib.request.urlopen(history_req)
        history_data = json.loads(history_res.read())
        if prompt_id in history_data:
            outputs = history_data[prompt_id]['outputs']
            break
    except:
        pass
    time.sleep(1)

print("\n🖼️ Decoding Upscaled Masterpiece:")
generated_images = []
for node_id, node_output in outputs.items():
    if 'images' in node_output:
        for image in node_output['images']:
            filename = image['filename']
            img_path = os.path.join(WORKSPACE, "output", filename)
            generated_images.append(img_path)
            print(f"\033[92m✓ [@CoinNoin] Saved: {filename}\033[0m")
            display(IPImage(filename=img_path))

# Direct download trigger
if AUTO_DOWNLOAD and generated_images:
    print("\n📥 [@CoinNoin] Download Started...")
    for img_file in generated_images:
        files.download(img_file)